# Direct-transfer evaluation

Evaluates direct-transfer model predictions on the study task.

This notebook accompanies [Stigmatizing Language in Gender-Expansive Patient Records: Corpus Development, Disparity Analysis, and Natural Language Processing-Based Detection Study](https://www.jmir.org/2026/1/e91089).

## Data and execution requirements

- Clinical note text and MIMIC identifiers are not included in this repository.
- Run this notebook only in an environment authorized to access MIMIC-IV and the credentialed annotation release.
- Set `GEP_DATA_DIR`, `GEP_MODEL_DIR`, `GEP_RESULTS_DIR`, and `GEP_FIGURES_DIR` as needed. By default, repository-local directories are used.
- The notebook outputs and execution counters have been removed from the public version.


In [ ]:
# Repository-local path configuration
from pathlib import Path
import os

PROJECT_ROOT = Path(os.environ.get("GEP_PROJECT_ROOT", Path.cwd())).resolve()
DATA_DIR = Path(os.environ.get("GEP_DATA_DIR", PROJECT_ROOT / "data")).resolve()
MODEL_DIR = Path(os.environ.get("GEP_MODEL_DIR", PROJECT_ROOT / "models")).resolve()
RESULTS_DIR = Path(os.environ.get("GEP_RESULTS_DIR", PROJECT_ROOT / "results")).resolve()
FIGURES_DIR = Path(os.environ.get("GEP_FIGURES_DIR", PROJECT_ROOT / "figures")).resolve()

for directory in (MODEL_DIR, RESULTS_DIR, FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from transformers import (
    LongformerTokenizer, LongformerForSequenceClassification,
    BertTokenizer, BertForSequenceClassification,
    AutoTokenizer, AutoModelForSequenceClassification
)
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, precision_score, recall_score

In [ ]:
# --- Longformer Dataset ---
class LongformerTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=4096, use_global_attention=True, global_attention_target='cls', dynamic_padding=False):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.global_attention_target = global_attention_target
        self.dynamic_padding = dynamic_padding
        self.use_global_attention = use_global_attention

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        padding_strategy = "longest" if self.dynamic_padding else "max_length"
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, padding=padding_strategy, max_length=self.max_length)
        input_ids = inputs["input_ids"].squeeze(0)
        attention_mask = inputs["attention_mask"].squeeze(0)
        output = {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': torch.tensor(label, dtype=torch.float)
        }
        if self.use_global_attention:
            global_attention_mask = torch.zeros_like(input_ids)
            if self.global_attention_target == 'cls':
                global_attention_mask[0] = 1
            output['global_attention_mask'] = global_attention_mask
        return output

def get_longformer_predictions(model, data_loader, device):
    model.eval()
    predictions, all_probs = [], []
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Predicting"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            # labels = batch['labels'].to(device)  # not used for prediction
            global_attention_mask = batch.get('global_attention_mask', None)
            if global_attention_mask is not None:
                global_attention_mask = global_attention_mask.to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, global_attention_mask=global_attention_mask)
            probabilities = torch.sigmoid(outputs.logits)
            predicted = (probabilities >= 0.5).float()
            predictions.extend(predicted.view(-1).cpu().numpy())
            all_probs.extend(probabilities.view(-1).cpu().numpy())
    return np.array(predictions), np.array(all_probs)

# --- Model configs: ONLY Longformer ---
longformer_model_configs = [
    {
        "name": "Longformer_MIMIC",
        "model_dir": str(MODEL_DIR / 'mimic_no_preprocess_no_glob_binary'),
        "tokenizer_cls": LongformerTokenizer,
        "model_cls": LongformerForSequenceClassification,
        "max_length": 4096
    },
    {
        "name": "Longformer_Berkeley_MIMIC",
        "model_dir": str(MODEL_DIR / 'new_berkeley_pretrain_to_mimic_v010725'),
        "tokenizer_cls": LongformerTokenizer,
        "model_cls": LongformerForSequenceClassification,
        "max_length": 4096
    },
    {
        "name": "Longformer_Berkeley_Phenotype_MIMIC",
        "model_dir": str(MODEL_DIR / 'new_berkeley_pretrain_to_phenotype_to_mimic_v030325'),
        "tokenizer_cls": LongformerTokenizer,
        "model_cls": LongformerForSequenceClassification,
        "max_length": 4096
    }
]

In [ ]:
mimic_train = pd.read_csv(str(DATA_DIR / 'GEP_train_80_20.csv'))
mimic_test  = pd.read_csv(str(DATA_DIR / 'GEP_test_80_20.csv'))
mimic_test = pd.concat([mimic_train, mimic_test], ignore_index=True)

In [ ]:
mimic_test.columns

In [ ]:
test_texts = mimic_test['text'].tolist()
test_labels = mimic_test['label'].tolist()

In [ ]:
# === Evaluation by overall sample and GEP group ===
import math
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# ---- Assumptions (must exist in your environment) ----
# - longformer_model_configs: list of dicts with keys ["name","model_dir","tokenizer_cls","model_cls","max_length"]
# - LongformerTextDataset: your dataset class returning dicts: 'input_ids','attention_mask', optional 'global_attention_mask'
# - test_texts, test_labels: sequences aligned with mimic_test rows
# - mimic_test: a DataFrame containing a 'GEP' column with values {0,1}
# ------------------------------------------------------

BATCH_SIZE = 8
NUM_WORKERS = 0
PIN_MEMORY = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def compute_metrics(y_true, y_pred, y_prob):
    """Compute metrics safely; AUC = NaN if only one class present."""
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_true, y_prob) if len(set(y_true)) > 1 else float("nan")
    return {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1 Score": f1, "AUC": auc}

all_rows = []

for config in longformer_model_configs:
    model_name    = config["name"]
    model_dir     = config["model_dir"]
    tokenizer_cls = config["tokenizer_cls"]
    model_cls     = config["model_cls"]
    max_length    = config["max_length"]

    print(f"\nLoading and evaluating model: {model_name}")
    try:
        tokenizer = tokenizer_cls.from_pretrained(model_dir)
        model = model_cls.from_pretrained(model_dir)
        model.to(device)
        model.eval()

        test_dataset = LongformerTextDataset(test_texts, test_labels, tokenizer, max_length=max_length)
        test_loader = DataLoader(
            test_dataset,
            batch_size=BATCH_SIZE,
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY
        )

        probs_list, preds_list = [], []

        # Single progress bar per model
        pbar = tqdm(total=len(test_loader), desc=f"{model_name}: predicting", unit="batch", leave=True)
        with torch.inference_mode():
            for batch in test_loader:
                try:
                    input_ids = batch["input_ids"].to(device, non_blocking=True)
                    attention_mask = batch["attention_mask"].to(device, non_blocking=True)
                    global_attention_mask = batch.get("global_attention_mask", None)
                    if global_attention_mask is not None:
                        global_attention_mask = global_attention_mask.to(device, non_blocking=True)

                    outputs = model(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        global_attention_mask=global_attention_mask
                    )
                    logits = outputs.logits
                    probs = torch.sigmoid(logits).view(-1).detach().cpu().numpy()
                    preds = (probs >= 0.5).astype(np.float32)

                    probs_list.extend(probs.tolist())
                    preds_list.extend(preds.tolist())

                except Exception as e:
                    print(f"  [Warn] Skipping a batch due to error: {e}")
                finally:
                    pbar.update(1)
        pbar.close()

        # Convert to arrays and hard-align lengths
        probs_arr = np.asarray(probs_list)
        preds_arr = np.asarray(preds_list)
        y_true_arr = np.asarray(test_labels)

        n_shared = min(len(y_true_arr), len(preds_arr), len(probs_arr), len(mimic_test))
        y_true_arr = y_true_arr[:n_shared]
        preds_arr  = preds_arr[:n_shared]
        probs_arr  = probs_arr[:n_shared]

        # ---- Overall metrics ----
        overall = compute_metrics(y_true_arr, preds_arr, probs_arr)
        all_rows.append({
            "Model": model_name,
            "Group": "Overall",
            "N": int(len(y_true_arr)),
            **overall
        })

        # ---- GEP-stratified metrics (strictly 0/1, direct indexing) ----
        if "GEP" not in mimic_test.columns:
            print("  [Warn] 'GEP' column not found in mimic_test; skipping subgroup analysis.")
            for g in (0, 1):
                all_rows.append({
                    "Model": model_name, "Group": f"GEP={g}", "N": 0,
                    "Accuracy": np.nan, "Precision": np.nan, "Recall": np.nan,
                    "F1 Score": np.nan, "AUC": np.nan
                })
        else:
            gep = mimic_test["GEP"].to_numpy()[:n_shared]
            # Ensure integer 0/1 (in case dtype is object/bool)
            try:
                gep = gep.astype(int)
            except Exception:
                # Fallback map if weird types appear
                gep = pd.Series(gep).map(lambda x: 1 if str(x).strip().lower() in ["1","true","yes"] else 0).to_numpy()

            # Optional sanity print
            print(f"  Subgroup counts -> GEP=0: {(gep==0).sum()}, GEP=1: {(gep==1).sum()}")

            # GEP=0
            idx0 = np.where(gep == 0)[0]
            if idx0.size:
                mets0 = compute_metrics(y_true_arr[idx0], preds_arr[idx0], probs_arr[idx0])
                all_rows.append({"Model": model_name, "Group": "GEP=0", "N": int(idx0.size), **mets0})
            else:
                all_rows.append({"Model": model_name, "Group": "GEP=0", "N": 0,
                                 "Accuracy": np.nan, "Precision": np.nan, "Recall": np.nan,
                                 "F1 Score": np.nan, "AUC": np.nan})

            # GEP=1
            idx1 = np.where(gep == 1)[0]
            if idx1.size:
                mets1 = compute_metrics(y_true_arr[idx1], preds_arr[idx1], probs_arr[idx1])
                all_rows.append({"Model": model_name, "Group": "GEP=1", "N": int(idx1.size), **mets1})
            else:
                all_rows.append({"Model": model_name, "Group": "GEP=1", "N": 0,
                                 "Accuracy": np.nan, "Precision": np.nan, "Recall": np.nan,
                                 "F1 Score": np.nan, "AUC": np.nan})

    except Exception as e:
        print(f"[Error] Loading or processing model {model_name}: {e}")
        all_rows.append({
            "Model": model_name,
            "Group": "Error",
            "N": np.nan,
            "Accuracy": np.nan, "Precision": np.nan, "Recall": np.nan,
            "F1 Score": np.nan, "AUC": np.nan,
            "Error": str(e)
        })

# ---- Results DataFrame + Save ----
results_df = pd.DataFrame(all_rows)

# Consistent column order
metric_cols = ["Model", "Group", "N", "Accuracy", "Precision", "Recall", "F1 Score", "AUC", "Error"]
for col in metric_cols:
    if col not in results_df.columns:
        results_df[col] = np.nan
results_df = results_df[metric_cols]

# Export
results_df.to_csv(str(RESULTS_DIR / 'model_performance_by_GEP.csv'), index=False)
results_df.to_excel(str(RESULTS_DIR / 'model_performance_by_GEP.xlsx'), index=False)

print("\n Results saved to:")
print(" - model_performance_by_GEP.csv")
print(" - model_performance_by_GEP.xlsx")

# Quick preview (works in notebooks)
try:
    from IPython.display import display
    display(results_df.head(12))
except Exception:
    print(results_df.head(12))


In [ ]:
test_texts = mimic_test['text'].tolist()
test_labels = mimic_test['label_exclude_misgendering'].tolist()

In [ ]:
# === Evaluation by overall sample and GEP group ===
import math
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# ---- Assumptions (must exist in your environment) ----
# - longformer_model_configs: list of dicts with keys ["name","model_dir","tokenizer_cls","model_cls","max_length"]
# - LongformerTextDataset: your dataset class returning dicts: 'input_ids','attention_mask', optional 'global_attention_mask'
# - test_texts, test_labels: sequences aligned with mimic_test rows
# - mimic_test: a DataFrame containing a 'GEP' column with values {0,1}
# ------------------------------------------------------

BATCH_SIZE = 32
NUM_WORKERS = 0
PIN_MEMORY = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def compute_metrics(y_true, y_pred, y_prob):
    """Compute metrics safely; AUC = NaN if only one class present."""
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_true, y_prob) if len(set(y_true)) > 1 else float("nan")
    return {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1 Score": f1, "AUC": auc}

all_rows = []

for config in longformer_model_configs:
    model_name    = config["name"]
    model_dir     = config["model_dir"]
    tokenizer_cls = config["tokenizer_cls"]
    model_cls     = config["model_cls"]
    max_length    = config["max_length"]

    print(f"\nLoading and evaluating model: {model_name}")
    try:
        tokenizer = tokenizer_cls.from_pretrained(model_dir)
        model = model_cls.from_pretrained(model_dir)
        model.to(device)
        model.eval()

        test_dataset = LongformerTextDataset(test_texts, test_labels, tokenizer, max_length=max_length)
        test_loader = DataLoader(
            test_dataset,
            batch_size=BATCH_SIZE,
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY
        )

        probs_list, preds_list = [], []

        # Single progress bar per model
        pbar = tqdm(total=len(test_loader), desc=f"{model_name}: predicting", unit="batch", leave=True)
        with torch.inference_mode():
            for batch in test_loader:
                try:
                    input_ids = batch["input_ids"].to(device, non_blocking=True)
                    attention_mask = batch["attention_mask"].to(device, non_blocking=True)
                    global_attention_mask = batch.get("global_attention_mask", None)
                    if global_attention_mask is not None:
                        global_attention_mask = global_attention_mask.to(device, non_blocking=True)

                    outputs = model(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        global_attention_mask=global_attention_mask
                    )
                    logits = outputs.logits
                    probs = torch.sigmoid(logits).view(-1).detach().cpu().numpy()
                    preds = (probs >= 0.5).astype(np.float32)

                    probs_list.extend(probs.tolist())
                    preds_list.extend(preds.tolist())

                except Exception as e:
                    print(f"  [Warn] Skipping a batch due to error: {e}")
                finally:
                    pbar.update(1)
        pbar.close()

        # Convert to arrays and hard-align lengths
        probs_arr = np.asarray(probs_list)
        preds_arr = np.asarray(preds_list)
        y_true_arr = np.asarray(test_labels)

        n_shared = min(len(y_true_arr), len(preds_arr), len(probs_arr), len(mimic_test))
        y_true_arr = y_true_arr[:n_shared]
        preds_arr  = preds_arr[:n_shared]
        probs_arr  = probs_arr[:n_shared]

        # ---- Overall metrics ----
        overall = compute_metrics(y_true_arr, preds_arr, probs_arr)
        all_rows.append({
            "Model": model_name,
            "Group": "Overall",
            "N": int(len(y_true_arr)),
            **overall
        })

        # ---- GEP-stratified metrics (strictly 0/1, direct indexing) ----
        if "GEP" not in mimic_test.columns:
            print("  [Warn] 'GEP' column not found in mimic_test; skipping subgroup analysis.")
            for g in (0, 1):
                all_rows.append({
                    "Model": model_name, "Group": f"GEP={g}", "N": 0,
                    "Accuracy": np.nan, "Precision": np.nan, "Recall": np.nan,
                    "F1 Score": np.nan, "AUC": np.nan
                })
        else:
            gep = mimic_test["GEP"].to_numpy()[:n_shared]
            # Ensure integer 0/1 (in case dtype is object/bool)
            try:
                gep = gep.astype(int)
            except Exception:
                # Fallback map if weird types appear
                gep = pd.Series(gep).map(lambda x: 1 if str(x).strip().lower() in ["1","true","yes"] else 0).to_numpy()

            # Optional sanity print
            print(f"  Subgroup counts -> GEP=0: {(gep==0).sum()}, GEP=1: {(gep==1).sum()}")

            # GEP=0
            idx0 = np.where(gep == 0)[0]
            if idx0.size:
                mets0 = compute_metrics(y_true_arr[idx0], preds_arr[idx0], probs_arr[idx0])
                all_rows.append({"Model": model_name, "Group": "GEP=0", "N": int(idx0.size), **mets0})
            else:
                all_rows.append({"Model": model_name, "Group": "GEP=0", "N": 0,
                                 "Accuracy": np.nan, "Precision": np.nan, "Recall": np.nan,
                                 "F1 Score": np.nan, "AUC": np.nan})

            # GEP=1
            idx1 = np.where(gep == 1)[0]
            if idx1.size:
                mets1 = compute_metrics(y_true_arr[idx1], preds_arr[idx1], probs_arr[idx1])
                all_rows.append({"Model": model_name, "Group": "GEP=1", "N": int(idx1.size), **mets1})
            else:
                all_rows.append({"Model": model_name, "Group": "GEP=1", "N": 0,
                                 "Accuracy": np.nan, "Precision": np.nan, "Recall": np.nan,
                                 "F1 Score": np.nan, "AUC": np.nan})

    except Exception as e:
        print(f"[Error] Loading or processing model {model_name}: {e}")
        all_rows.append({
            "Model": model_name,
            "Group": "Error",
            "N": np.nan,
            "Accuracy": np.nan, "Precision": np.nan, "Recall": np.nan,
            "F1 Score": np.nan, "AUC": np.nan,
            "Error": str(e)
        })

# ---- Results DataFrame + Save ----
results_df = pd.DataFrame(all_rows)

# Consistent column order
metric_cols = ["Model", "Group", "N", "Accuracy", "Precision", "Recall", "F1 Score", "AUC", "Error"]
for col in metric_cols:
    if col not in results_df.columns:
        results_df[col] = np.nan
results_df = results_df[metric_cols]

# Export
results_df.to_csv(str(RESULTS_DIR / 'model_performance_by_GEP_exclude_misgender.csv'), index=False)
results_df.to_excel(str(RESULTS_DIR / 'model_performance_by_GEP_exclude_misgender.xlsx'), index=False)

print("\n Results saved.")

# Quick preview (works in notebooks)
try:
    from IPython.display import display
    display(results_df.head(12))
except Exception:
    print(results_df.head(12))
